In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc

from PIL import Image
import matplotlib
matplotlib.use("Agg")  # 서버/GUI 없는 환경에서도 저장 가능
import matplotlib.pyplot as plt

import torch

# open_clip, umap (필수)
try:
    import open_clip
except ImportError as e:
    raise ImportError("open_clip_torch가 필요합니다. 설치: pip install open_clip_torch") from e

try:
    import umap
except ImportError as e:
    raise ImportError("umap-learn이 필요합니다. 설치: pip install umap-learn") from e

print("Imports OK")

In [ ]:
# ====== 환경설정 (여기만 수정) ======
BASE_DIR = "/home/ice06/project/HS/Dataset_0223/Case2"
OUTPUT_DIR = "/home/ice06/project/HS/0215/0226_clip_umap"

#============폴더명은 TP, FP, TP, FN으로 설정 ====================
DATASET_CONFIG = {
    "Real_TN": ["TN"], 
    "Real_FP": ["FP"],
    "Fake_TP": ["TP"],
    "Fake_FN": ["FN"],
}

IMAGES_PER_GROUP = 100000   
CLIP_MODEL = "ViT-B-32"
DEVICE = "cuda"            # "cuda" 또는 "cpu"

In [ ]:
from pathlib import Path

def assert_exists(p, name):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f" {name} not found: {p}")
    print(f" {name}: {p}")
    return p

assert_exists(BASE_DIR, "BASE_DIR")
for group, folders in DATASET_CONFIG.items():
    for folder in folders:
        assert_exists(Path(BASE_DIR) / folder, f"Dataset folder ({group})")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(" OUTPUT_DIR ready:", OUTPUT_DIR)

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff")

def clear_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()


class ClipEmbedder:
    def __init__(self, model_name="ViT-B-32", device="cuda"):
        self.device = device if (device == "cpu" or torch.cuda.is_available()) else "cpu"
        print(f" CLIP 모델 로드 중 ({model_name})... 장치: {self.device}")
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name, pretrained="openai"
        )
        self.model.eval().to(self.device)

    @torch.no_grad()
    def encode(self, path: str) -> np.ndarray:
        img = Image.open(path).convert("RGB")
        x = self.preprocess(img).unsqueeze(0).to(self.device)
        feat = self.model.encode_image(x)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        return feat.squeeze(0).cpu().numpy().astype(np.float32)


def umap_plot(X2: np.ndarray, labels: np.ndarray, out_path: str, title: str):
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "grid.linestyle": "--",
    })

    fig, ax = plt.subplots(figsize=(10, 8), dpi=200)
    ax.set_facecolor("white")

    color_map = {
        "Real_TN": "#2ecc71",
        "Real_FP": "#f39c12",
        "Fake_TP": "#3498db",
        "Fake_FN": "#e74c3c",
    }
    marker_map = {
        "Real_TN": "o",
        "Real_FP": "D",
        "Fake_TP": "^",
        "Fake_FN": "s",
    }

    groups = ["Real_TN", "Real_FP", "Fake_TP", "Fake_FN"]

    for group in groups:
        idx = (labels == group)
        if not np.any(idx):
            continue

        ax.scatter(
            X2[idx, 0], X2[idx, 1],
            s=40,
            alpha=0.7,
            c=color_map.get(group, "#95a5a6"),
            marker=marker_map.get(group, "o"),
            edgecolors="white",
            linewidths=0.5,
            label=f"{group} (n={np.sum(idx)})"
        )

    ax.set_title(title, pad=15)
    ax.set_xlabel("UMAP Dimension 1")
    ax.set_ylabel("UMAP Dimension 2")

    for spine in ax.spines.values():
        spine.set_edgecolor("#dddddd")

    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        frameon=True,
        facecolor="white",
        edgecolor="#cccccc",
        title="Dataset Group",
        title_fontsize=11
    )

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"저장 완료: {out_path}")

In [ ]:
def run_clip_analysis(
    base_dir,
    out_dir,
    dataset_config,
    images_per_group=1000,
    clip_model="ViT-B-32",
    device="cuda"
):
    os.makedirs(out_dir, exist_ok=True)

    print("\n 1. 이미지 경로 로드 중...")
    paths, labels = [], []
    groups = list(dataset_config.keys())

    for group in groups:
        folders = dataset_config[group]
        group_paths = []

        for folder in folders:
            folder_path = os.path.join(base_dir, folder)
            if os.path.exists(folder_path):
                for ext in IMG_EXT:
                    group_paths.extend(glob.glob(os.path.join(folder_path, f"*{ext}")))
                    group_paths.extend(glob.glob(os.path.join(folder_path, f"*{ext.upper()}")))
            else:
                print(f"폴더 없음: {folder_path}")

        group_paths = list(set(group_paths))[:images_per_group]
        paths.extend(group_paths)
        labels.extend([group] * len(group_paths))
        print(f"  {group}: {len(group_paths)} images")

    if len(paths) == 0:
        print(" 처리할 이미지가 없습니다. 경로를 확인해주세요.")
        return None

    print(f"\n 2. CLIP 특징 추출 시작 (총 {len(paths)}장)...")
    embedder = ClipEmbedder(clip_model, device)

    embs, valid_paths, valid_labels = [], [], []

    for path, label in tqdm(zip(paths, labels), total=len(paths), desc="Extracting"):
        try:
            feat = embedder.encode(path)
            embs.append(feat)
            valid_paths.append(path)
            valid_labels.append(label)
        except Exception as e:
            print(f"\n파일 처리 실패 ({path}): {e}")
            continue

    clear_gpu_memory()

    if len(embs) < 2:
        print(" UMAP을 계산하기에 유효한 이미지가 너무 적습니다.")
        return None

    E = np.stack(embs, axis=0)
    L = np.array(valid_labels)

    print("\n 3. UMAP 차원 축소 계산 중...")
    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=42
    )
    E2 = reducer.fit_transform(E)

    print("\n 4. 시각화 및 CSV 저장...")
    out_img = os.path.join(out_dir, "clip_umap_visualization.png")
    umap_plot(E2, L, out_img, title="UMAP Visualization of CLIP Embeddings (4 Classes)")

    csv_path = os.path.join(out_dir, "clip_umap_coordinates.csv")
    df = pd.DataFrame({
        "File_Path": valid_paths,
        "Group": L,
        "UMAP_X": E2[:, 0],
        "UMAP_Y": E2[:, 1]
    })
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f" CSV 저장 완료: {csv_path}")

    print(f"\n 완료! 결과 폴더: {out_dir}")
    return {
        "embeddings": E,
        "umap_2d": E2,
        "labels": L,
        "paths": valid_paths,
        "out_img": out_img,
        "csv_path": csv_path
    }

In [ ]:
results = run_clip_analysis(
    base_dir=BASE_DIR,
    out_dir=OUTPUT_DIR,
    dataset_config=DATASET_CONFIG,
    images_per_group=IMAGES_PER_GROUP,
    clip_model=CLIP_MODEL,
    device=DEVICE
)
results